# Coding Challenges: Trickle Up (Lead Full Stack)

These challenges are designed to simulate real-world problems relevant to high-performance microservices and frontend data processing in the Oil & Gas industry.

## Challenge 1: Implementation of a Circuit Breaker

**Problem:** Implement a simple `CircuitBreaker` class in Python that wraps a function call. It should have three states: `CLOSED`, `OPEN`, and `HALF_OPEN`. 
- If a function fails $N$ times, the circuit opens for $T$ seconds.
- While open, all calls fail immediately.
- After $T$ seconds, it enters `HALF_OPEN` where a single success resets it to `CLOSED`.

In [ ]:
import time
from enum import Enum

class State(Enum):
    CLOSED = 1
    OPEN = 2
    HALF_OPEN = 3

class CircuitBreaker:
    def __init__(self, failure_threshold=3, recovery_timeout=5):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.state = State.CLOSED
        self.failures = 0
        self.last_failure_time = None

    def call(self, func, *args, **kwargs):
        if self.state == State.OPEN:
            if time.time() - self.last_failure_time > self.recovery_timeout:
                self.state = State.HALF_OPEN
            else:
                raise Exception("Circuit is OPEN")

        try:
            result = func(*args, **kwargs)
            self._on_success()
            return result
        except Exception as e:
            self._on_failure()
            raise e

    def _on_success(self):
        self.failures = 0
        self.state = State.CLOSED

    def _on_failure(self):
        self.failures += 1
        self.last_failure_time = time.time()
        if self.failures >= self.failure_threshold:
            self.state = State.OPEN

# Example usage
def unreliable_service():
    raise Exception("Network Error")

cb = CircuitBreaker(failure_threshold=2, recovery_timeout=2)

try:
    cb.call(unreliable_service)
except Exception as e: print(e)

try:
    cb.call(unreliable_service)
except Exception as e: print(e)

print(f"State after 2 failures: {cb.state}")

## Challenge 2: Processing High-Frequency Sensor Data

**Problem:** You are receiving a stream of sensor readings (timestamp, value). Implement a function to calculate a moving average over a window of $K$ seconds for a specific sensor ID.

In [ ]:
from collections import deque

class SensorProcessor:
    def __init__(self, window_seconds=60):
        self.window_seconds = window_seconds
        self.readings = deque()
        self.current_sum = 0.0

    def add_reading(self, timestamp, value):
        self.readings.append((timestamp, value))
        self.current_sum += value
        
        # Remove readings outside the window
        while self.readings and timestamp - self.readings[0][0] > self.window_seconds:
            _, old_value = self.readings.popleft()
            self.current_sum -= old_value

    def get_average(self):
        if not self.readings:
            return 0.0
        return self.current_sum / len(self.readings)

# Example
processor = SensorProcessor(window_seconds=10)
processor.add_reading(100, 50)
processor.add_reading(105, 60)
processor.add_reading(111, 70) # 100 is now outside window (111-100 > 10)
print(f"Average: {processor.get_average()}") # Should average 60 and 70